# Tamil ASR model comparison

Transcribes the SAME 2-min call clip with 3 models and prints all
transcripts for side-by-side quality judgement:
1. `vasista22/whisper-tamil-large-v2` (attached as pre-converted CT2 dataset)
2. `deepdml/whisper-large-v3-turbo-ta-mix-norm` (code-mix tuned, 4-layer decoder)
3. `openai/whisper-large-v3` (vanilla baseline)

In [ ]:
!pip -q install faster-whisper ctranslate2 transformers soundfile

In [ ]:
import glob, subprocess, numpy as np, os

for root, dirs, files in os.walk('/kaggle/input'):
    print(root, '->', files[:8])
cands = [p for p in glob.glob('/kaggle/input/**/*.wav', recursive=True) +
         glob.glob('/kaggle/input/**/*.mp3', recursive=True)]
assert cands, 'no audio mounted'
clip = sorted(cands)[0]
print('clip:', clip)
r = subprocess.run(['ffmpeg','-v','error','-i',clip,'-f','f32le','-acodec','pcm_f32le','-ar','16000','-ac','1','-'],
                   capture_output=True, check=True)
AUDIO = np.frombuffer(r.stdout, dtype=np.float32)
print('decoded', len(AUDIO)/16000, 'sec')

In [ ]:
import os, subprocess, time
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

def ct2_dir(hf_id, name):
    '''Pre-converted model attached as dataset; otherwise download+convert.'''
    for attached in glob.glob(f'/kaggle/input/datasets/*/{name}') + [f'/kaggle/input/{name}']:
        if os.path.isfile(os.path.join(attached, 'model.bin')):
            return attached
    out = f'/kaggle/working/{name}'
    if not os.path.isfile(os.path.join(out, 'model.bin')):
        loc = f'/kaggle/working/hf-{name}'
        snapshot_download(hf_id, local_dir=loc,
                          allow_patterns=['*.json','*.txt','pytorch_model.bin','model.safetensors'])
        AutoTokenizer.from_pretrained(loc).save_pretrained(loc)
        subprocess.run(['ct2-transformers-converter','--model',loc,'--output_dir',out,
                        '--quantization','int8_float16'], check=True)
        AutoTokenizer.from_pretrained(loc).save_pretrained(out)
    return out

MODELS = [
    ('vasista22-tamil-large-v2', 'vasista22/whisper-tamil-large-v2', 'tamil-ct2-model'),
    ('deepdml-turbo-ta-mix',    'deepdml/whisper-large-v3-turbo-ta-mix-norm', 'ct2-deepdml'),
    ('vanilla-large-v3',        'openai/whisper-large-v3', 'ct2-vanilla'),
]
for name, hf, d in MODELS:
    print(name, '->', ct2_dir(hf, d))

In [ ]:
import time
from faster_whisper import WhisperModel

def run(model_dir, label):
    m = WhisperModel(model_dir, device='cuda', compute_type='int8_float16')
    t0 = time.time()
    segs, info = m.transcribe(AUDIO, language='ta', beam_size=5, vad_filter=True,
                              vad_parameters={'threshold':0.35,'min_silence_duration_ms':500,'speech_pad_ms':200})
    lines = [f'[{s.start:6.1f}-{s.end:6.1f}] {s.text}' for s in segs]
    dt = time.time()-t0
    del m
    import gc, torch; gc.collect(); torch.cuda.empty_cache()
    print('='*80); print(f'{label}  |  {dt:.1f}s wall | lang={info.language}')
    print('='*80); print('\n'.join(lines)); print()
    return lines

results = {}
for name, hf, d in MODELS:
    path = ct2_dir(hf, d)
    try: results[name] = run(path, name)
    except Exception as e: print(name, 'FAILED:', e)

with open('/kaggle/working/model_comparison.txt','w',encoding='utf-8') as f:
    for name, lines in results.items():
        f.write('='*80+'\n'+name+'\n'+'='*80+'\n'+'\n'.join(lines)+'\n\n')
print('saved model_comparison.txt')

In [ ]:
# don't let multi-GB converted models inflate the kernel output download
import shutil, glob
for d in glob.glob('/kaggle/working/ct2-*') + glob.glob('/kaggle/working/hf-*'):
    shutil.rmtree(d, ignore_errors=True)
    print('removed', d)